In [1]:
import pandas as pd 
import unicodedata 
import re

In [2]:
bill_details= pd.read_csv('../data/rawdata/MA_Bill_Details_raw.csv')
legislature_members= pd.read_csv('../data/rawdata/MA_legislature_members_raw.csv')
roll_calls= pd.read_csv('../data/rawdata/MA_Senate_rollcalls_raw.csv')
bill_rollcalls= pd.read_csv('../data/rawdata/MA_Bill_rollcall_mapping_raw.csv')


In [3]:
display(bill_details.head())
display(legislature_members.head())
display(roll_calls.head())
display(bill_rollcalls.head())

,GeneralCourtNumber,BillNumber,DocketNumber,PrimarySponsorId,PrimarySponsorName,Title
0,183,NaN,SD2779,NaN,NaN,A report of the Executive Office of Transporta...
1,183,NaN,SD2780,NaN,NaN,Communication from the Hampden County Sheriff’...
2,183,NaN,SD2781,NaN,NaN,Communication from the Hampshire County Sherif...
3,183,NaN,SD2782,NaN,NaN,A report of the Special Transportation Finance...
4,183,NaN,SD2786,NaN,NaN,A report of the Massachusetts School Building ...


,GeneralCourtNumber,MemberCode,FullName,District,Branch,Party,Active,LeadershipPosition
0,194,DAF0,Dylan A. Fernandes,Plymouth and Barnstable,Senate,Democrat,NaN,NaN
1,194,KAD0,Kelly A. Dooner,Third Bristol and Plymouth,Senate,Republican,NaN,Assistant Minority Leader
2,194,WJD0,"William J. Driscoll, Jr.","Norfolk, Plymouth and Bristol",Senate,Democrat,NaN,NaN
3,194,MLB1,Michelle L. Badger,1st Plymouth,House,Democrat,NaN,NaN
4,194,MSC1,Michael S. Chaisson,1st Bristol,House,Republican,NaN,NaN


,GeneralCourtNumber,Branch,RollCallNumber,MemberCode,MemberURL,Vote
0,188,Senate,2,MJB0,http://malegislature.gov/api/GeneralCourts/188...,Yeas
1,188,Senate,2,SMB0,http://malegislature.gov/api/GeneralCourts/188...,Yeas
2,188,Senate,2,WNB0,http://malegislature.gov/api/GeneralCourts/188...,Yeas
3,188,Senate,2,GDC0,http://malegislature.gov/api/GeneralCourts/188...,Yeas
4,188,Senate,2,HLC0,http://malegislature.gov/api/GeneralCourts/188...,Yeas


,GeneralCourtNumber,BillNumber,RollCallNumber
0,186,H1696,293
1,186,H4156,348
2,186,H4316,295
3,186,H4368,291
4,186,H4470,190


## Data Cleaning

### Bills

In [4]:
bill_details.head()

,GeneralCourtNumber,BillNumber,DocketNumber,PrimarySponsorId,PrimarySponsorName,Title
0,183,NaN,SD2779,NaN,NaN,A report of the Executive Office of Transporta...
1,183,NaN,SD2780,NaN,NaN,Communication from the Hampden County Sheriff’...
2,183,NaN,SD2781,NaN,NaN,Communication from the Hampshire County Sherif...
3,183,NaN,SD2782,NaN,NaN,A report of the Special Transportation Finance...
4,183,NaN,SD2786,NaN,NaN,A report of the Massachusetts School Building ...


In [5]:
bill_details.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80271 entries, 0 to 80270
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   GeneralCourtNumber  80271 non-null  int64 
 1   BillNumber          69214 non-null  object
 2   DocketNumber        72449 non-null  object
 3   PrimarySponsorId    70200 non-null  object
 4   PrimarySponsorName  79216 non-null  object
 5   Title               80050 non-null  object
dtypes: int64(1), object(5)
memory usage: 3.7+ MB


In [6]:
bill_details['GeneralCourtNumber'].value_counts()

GeneralCourtNumber
192    10363
193    10156
191     9690
190     9021
189     8999
194     8785
188     8278
186     7789
187     7176
183        9
185        3
184        2
Name: count, dtype: int64

In [7]:

bill_details_clean = bill_details.copy()

# Clean text fields in bill_details by converting to string, stripping whitespace, and replacing 'nan' with pd.NA
for col in ['BillNumber', 'DocketNumber', 'PrimarySponsorId', 'PrimarySponsorName', 'Title']:
    bill_details_clean[col] = (
        bill_details_clean[col]
        .astype(str)
        .str.strip()
        .replace({'nan': pd.NA})
    )

# Remove duplicate rows
bill_details_clean = bill_details_clean.drop_duplicates()

# Remove rows missing all major identifiers
bill_details_clean = bill_details_clean.dropna(subset=['BillNumber'], how='all')

# Convert GeneralCourtNumber to integer type
bill_details_clean['GeneralCourtNumber'] = bill_details_clean['GeneralCourtNumber'].astype(int)

print(bill_details_clean.info())


<class 'pandas.core.frame.DataFrame'>
Index: 69214 entries, 9 to 80267
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   GeneralCourtNumber  69214 non-null  int64 
 1   BillNumber          69214 non-null  object
 2   DocketNumber        61392 non-null  object
 3   PrimarySponsorId    67566 non-null  object
 4   PrimarySponsorName  68618 non-null  object
 5   Title               69181 non-null  object
dtypes: int64(1), object(5)
memory usage: 3.7+ MB
None


In [8]:
bill_details_clean.head()

,GeneralCourtNumber,BillNumber,DocketNumber,PrimarySponsorId,PrimarySponsorName,Title
9,184,H4176,HD94176,<NA>,<NA>,relative to HIV and Hepatitis C prevention
11,185,S2742,SD92742,<NA>,<NA>,An Act relative to landscape sprinkler systems
12,185,S2783,SD92783,S48,Senate Committee on Post Audit and Oversight,A report of the committee on Post Audit and Ov...
13,185,S2870,SD92870,S48,Senate Committee on Post Audit and Oversight,The Cost of Inaction: Climate Change in the Co...
14,186,S99,SD1,BET0,Bruce E. Tarr,An Act to provide for loans for waterfront dev...


In [9]:
bill_details_clean.to_csv('MA_bill_details.csv', index=False)

### Legislature members

In [10]:
legislature_members.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2773 entries, 0 to 2772
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   GeneralCourtNumber  2773 non-null   int64  
 1   MemberCode          2773 non-null   object 
 2   FullName            2768 non-null   object 
 3   District            2143 non-null   object 
 4   Branch              2769 non-null   object 
 5   Party               2662 non-null   object 
 6   Active              0 non-null      float64
 7   LeadershipPosition  369 non-null    object 
dtypes: float64(1), int64(1), object(6)
memory usage: 173.4+ KB


In [11]:
legislature_members.head()

,GeneralCourtNumber,MemberCode,FullName,District,Branch,Party,Active,LeadershipPosition
0,194,DAF0,Dylan A. Fernandes,Plymouth and Barnstable,Senate,Democrat,NaN,NaN
1,194,KAD0,Kelly A. Dooner,Third Bristol and Plymouth,Senate,Republican,NaN,Assistant Minority Leader
2,194,WJD0,"William J. Driscoll, Jr.","Norfolk, Plymouth and Bristol",Senate,Democrat,NaN,NaN
3,194,MLB1,Michelle L. Badger,1st Plymouth,House,Democrat,NaN,NaN
4,194,MSC1,Michael S. Chaisson,1st Bristol,House,Republican,NaN,NaN


In [12]:
# Checking if a member has inconsistent details across general courts for name, district, branch or party

cols_to_check = ['FullName', 'District', 'Branch', 'Party']
inconsistent_members = (
    legislature_members
    .groupby('MemberCode')[cols_to_check]
    .nunique()  
    .reset_index()
)
inconsistent_members = inconsistent_members[
    inconsistent_members[cols_to_check].gt(1).any(axis=1)
]
print(inconsistent_members)

    MemberCode  FullName  District  Branch  Party
16        AMG0         1         2       1      1
21        AMS2         2         1       1      1
25        AST1         2         1       1      1
27        AXP1         2         0       1      1
29        A_G0         2         1       1      1
73        CAR1         2         1       1      1
88        CMS1         3         1       1      1
94        CSC0         1         2       1      1
122       DFH1         4         1       1      1
149       E P1         2         0       1      1
183       GNP1         3         1       1      1
187       G_C1         2         1       1      1
203       JAH0         4         1       1      1
217       JCV0         1         2       1      1
220       JDP1         2         1       1      1
250       JML1         2         1       1      1
292       KES0         1         2       1      1
320       LDF1         2         1       1      1
325       LME0         1         2       1      1


In [13]:
# Listing details of members with inconsistent data
dup_names= inconsistent_members[inconsistent_members['FullName'].gt(1)]
for i in dup_names.values:
    print(legislature_members[legislature_members['MemberCode']==i[0]][['MemberCode', 'FullName']])

    MemberCode                    FullName
50        AMS2  Alyson M. Sullivan-Almeida
232       AMS2  Alyson M. Sullivan-Almeida
430       AMS2          Alyson M. Sullivan
858       AMS2  Alyson M. Sullivan-Almeida
     MemberCode          FullName
1910       AST1  A. Stephen Tobin
1936       AST1       A. S. Tobin
2266       AST1       A. S. Tobin
2382       AST1       A. S. Tobin
2652       AST1       A. S. Tobin
     MemberCode                FullName
1938       AXP1  Anthony W. Petruccelli
2267       AXP1     Anthony Petruccelli
2383       AXP1     Anthony Petruccelli
2653       AXP1     Anthony Petruccelli
    MemberCode    FullName
194       A_G0  Adam Gómez
400       A_G0  Adam Gomez
424       A_G0  Adam Gomez
     MemberCode                  FullName
1383       CAR1  Cheryl A. Coakley-Rivera
1538       CAR1  Cheryl A. Coakley-Rivera
1756       CAR1  Cheryl A. Coakley-Rivera
2284       CAR1  Cheryl A. Coakley-Rivera
2357       CAR1          Cheryl A. Rivera
2673       CAR1      

In [14]:
def clean_text(s):
    if pd.isna(s):
        return s
    # removing accent
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('utf-8', 'ignore')
    # removing punctuation marks such as '.', ',', '-'
    s = re.sub(r'[^\w\s]', '', s)
    # removing spaces and convert to title case
    s=re.sub(r'\s+', ' ', s).strip().title()
    return s

def text_normalize(df):
    cols_to_check = ['FullName', 'District', 'Branch', 'Party']

    # normalizing name - removing extra characters, replace name with longest when a member has multiple values in name
    df['FullName_clean'] = df['FullName'].apply(clean_text)
    df['FullName_length'] = df['FullName_clean'].fillna('').apply(len)
    longest_name = (
        df.loc[df.groupby('MemberCode')['FullName_length'].idxmax(), ['MemberCode', 'FullName_clean']]
        .rename(columns={'FullName_clean': 'LongestFullName'})
    )
    # Merge back — replace all names for that MemberCode
    df = df.merge(longest_name, on='MemberCode', how='left')
    df['FullName'] = df['LongestFullName']
    df.drop(columns=['FullName_clean', 'FullName_length', 'LongestFullName'], inplace=True)

# normalizing district
    df['District_clean'] = df['District'].apply(clean_text)
    df['District_length'] = df['District_clean'].fillna('').apply(len)
    longest_district = (
        df.loc[df.groupby('MemberCode')['District_length'].idxmax(), ['MemberCode', 'District_clean']]
        .rename(columns={'District_clean': 'LongestDistrict'})
    )
    # Merge back — replace all districts for that MemberCode
    df = df.merge(longest_district, on='MemberCode', how='left')
    df['District'] = df['LongestDistrict']
    df.drop(columns=['District_clean', 'District_length', 'LongestDistrict'], inplace=True)

    return df



legislature_members_clean = legislature_members.copy()

text_cols = ['MemberCode', 'FullName', 'District', 'Branch', 'Party', 'LeadershipPosition']
for col in text_cols:
    legislature_members_clean[col] = (
        legislature_members_clean[col]
        .astype(str)
        .str.strip()
        .replace({'nan': pd.NA, 'NaN': pd.NA})
    )
legislature_members_clean= text_normalize( legislature_members_clean)

legislature_members_clean= legislature_members_clean.drop('Active', axis=1)

legislature_members_clean = legislature_members_clean.drop_duplicates(subset=['GeneralCourtNumber', 'MemberCode'], keep='first')

legislature_members_clean['GeneralCourtNumber'] = legislature_members_clean['GeneralCourtNumber'].astype(int)

print(legislature_members_clean.info())



<class 'pandas.core.frame.DataFrame'>
Index: 2696 entries, 0 to 2772
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   GeneralCourtNumber  2696 non-null   int64 
 1   MemberCode          2696 non-null   object
 2   FullName            2692 non-null   object
 3   District            2336 non-null   object
 4   Branch              2692 non-null   object
 5   Party               2585 non-null   object
 6   LeadershipPosition  347 non-null    object
dtypes: int64(1), object(6)
memory usage: 168.5+ KB
None


In [15]:
print(legislature_members_clean['Branch'].value_counts())
print(legislature_members_clean['Party'].value_counts())
print(legislature_members_clean['LeadershipPosition'].value_counts())

Branch
House     2135
Senate     546
Joint       11
Name: count, dtype: int64
Party
Democrat       2152
Republican      417
Independent       9
Unenrolled        7
Name: count, dtype: int64
LeadershipPosition
Assistant Majority Leader             41
Second Assistant Majority Leader      37
Third Assistant Minority Leader       31
Minority Leader                       26
Majority Leader                       24
Fourth Division Chair                 19
Second Assistant Minority Leader      19
Assistant Minority Leader             18
Third Division Chair                  16
President of the Senate               15
House Ways and Means Chair            14
Senate President Emerita              10
Speaker of the House                   9
Senate Majority Whip                   9
Speaker Pro Tempore                    9
First Assistant Minority Leader        8
President Pro Tempore                  8
First Division Chair                   8
Second Division Chair                  7
Assistant Ma

In [16]:
legislature_members_clean.head()

,GeneralCourtNumber,MemberCode,FullName,District,Branch,Party,LeadershipPosition
0,194,DAF0,Dylan A Fernandes,Plymouth And Barnstable,Senate,Democrat,<NA>
1,194,KAD0,Kelly A Dooner,Third Bristol And Plymouth,Senate,Republican,Assistant Minority Leader
2,194,WJD0,William J Driscoll Jr,Norfolk Plymouth And Bristol,Senate,Democrat,<NA>
3,194,MLB1,Michelle L Badger,1St Plymouth,House,Democrat,<NA>
4,194,MSC1,Michael S Chaisson,1St Bristol,House,Republican,<NA>


In [17]:
legislature_members_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2696 entries, 0 to 2772
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   GeneralCourtNumber  2696 non-null   int64 
 1   MemberCode          2696 non-null   object
 2   FullName            2692 non-null   object
 3   District            2336 non-null   object
 4   Branch              2692 non-null   object
 5   Party               2585 non-null   object
 6   LeadershipPosition  347 non-null    object
dtypes: int64(1), object(6)
memory usage: 168.5+ KB


In [17]:
legislature_members_clean.to_csv('MA_legislature_members.csv', index=False)

### Roll calls

In [18]:
roll_calls.head()


,GeneralCourtNumber,Branch,RollCallNumber,MemberCode,MemberURL,Vote
0,188,Senate,2,MJB0,http://malegislature.gov/api/GeneralCourts/188...,Yeas
1,188,Senate,2,SMB0,http://malegislature.gov/api/GeneralCourts/188...,Yeas
2,188,Senate,2,WNB0,http://malegislature.gov/api/GeneralCourts/188...,Yeas
3,188,Senate,2,GDC0,http://malegislature.gov/api/GeneralCourts/188...,Yeas
4,188,Senate,2,HLC0,http://malegislature.gov/api/GeneralCourts/188...,Yeas


In [19]:
roll_calls.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99031 entries, 0 to 99030
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   GeneralCourtNumber  99031 non-null  int64 
 1   Branch              99031 non-null  object
 2   RollCallNumber      99031 non-null  int64 
 3   MemberCode          98851 non-null  object
 4   MemberURL           98851 non-null  object
 5   Vote                99031 non-null  object
dtypes: int64(2), object(4)
memory usage: 4.5+ MB


In [21]:
# Cleaning roll_calls data
roll_calls_clean = roll_calls.copy()

# Clean text fields in roll_calls by converting to string, stripping whitespace, and replacing 'nan' with pd.NA
string_cols = ['Branch', 'MemberCode', 'MemberURL', 'Vote']
for col in string_cols:
    roll_calls_clean[col] = (
        roll_calls_clean[col]
        .astype(str)
        .str.strip()
        .replace({'nan': pd.NA, 'NaN': pd.NA})
    )

# Normalizing Vote values 
roll_calls_clean['Vote'] = (
    roll_calls_clean['Vote']
    .replace({
        'Yeas': 'Y',
        'Nays':'N',
        'Present': 'P',
        'Absent': 'A'
    })
)

# Remove duplicate rows
roll_calls_clean = roll_calls_clean.drop_duplicates(subset=['GeneralCourtNumber', 'Branch', 'RollCallNumber', 'MemberCode'])

# Convert GeneralCourtNumber and RollCallNumber to integer type
roll_calls_clean['GeneralCourtNumber'] = roll_calls_clean['GeneralCourtNumber'].astype(int)
roll_calls_clean['RollCallNumber'] = roll_calls_clean['RollCallNumber'].astype(int)

print(roll_calls_clean.info())


<class 'pandas.core.frame.DataFrame'>
Index: 98991 entries, 0 to 99030
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   GeneralCourtNumber  98991 non-null  int64 
 1   Branch              98991 non-null  object
 2   RollCallNumber      98991 non-null  int64 
 3   MemberCode          98811 non-null  object
 4   MemberURL           98811 non-null  object
 5   Vote                98991 non-null  object
dtypes: int64(2), object(4)
memory usage: 5.3+ MB
None


In [22]:
roll_calls_clean.drop(columns=['MemberURL'], inplace=True)

In [23]:
roll_calls_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 98991 entries, 0 to 99030
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   GeneralCourtNumber  98991 non-null  int64 
 1   Branch              98991 non-null  object
 2   RollCallNumber      98991 non-null  int64 
 3   MemberCode          98811 non-null  object
 4   Vote                98991 non-null  object
dtypes: int64(2), object(3)
memory usage: 4.5+ MB


In [24]:
roll_calls_clean.to_csv('MA_senate_rollcalls.csv', index=False)

In [25]:
roll_calls_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 98991 entries, 0 to 99030
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   GeneralCourtNumber  98991 non-null  int64 
 1   Branch              98991 non-null  object
 2   RollCallNumber      98991 non-null  int64 
 3   MemberCode          98811 non-null  object
 4   Vote                98991 non-null  object
dtypes: int64(2), object(3)
memory usage: 4.5+ MB


In [22]:
roll_calls_clean.head()

,GeneralCourtNumber,Branch,RollCallNumber,MemberCode,MemberURL,Vote
0,188,Senate,2,MJB0,http://malegislature.gov/api/GeneralCourts/188...,Y
1,188,Senate,2,SMB0,http://malegislature.gov/api/GeneralCourts/188...,Y
2,188,Senate,2,WNB0,http://malegislature.gov/api/GeneralCourts/188...,Y
3,188,Senate,2,GDC0,http://malegislature.gov/api/GeneralCourts/188...,Y
4,188,Senate,2,HLC0,http://malegislature.gov/api/GeneralCourts/188...,Y


In [23]:
bill_details_clean.head()

,GeneralCourtNumber,BillNumber,DocketNumber,PrimarySponsorId,PrimarySponsorName,Title
9,184,H4176,HD94176,<NA>,<NA>,relative to HIV and Hepatitis C prevention
11,185,S2742,SD92742,<NA>,<NA>,An Act relative to landscape sprinkler systems
12,185,S2783,SD92783,S48,Senate Committee on Post Audit and Oversight,A report of the committee on Post Audit and Ov...
13,185,S2870,SD92870,S48,Senate Committee on Post Audit and Oversight,The Cost of Inaction: Climate Change in the Co...
14,186,S99,SD1,BET0,Bruce E. Tarr,An Act to provide for loans for waterfront dev...


In [ ]:
# Checking for missing roll call numbers in each general court

# Identify maximum roll call number per general court and compare with existing roll call numbers
max_rollcalls= roll_calls_clean.groupby('GeneralCourtNumber')['RollCallNumber'].max()
court_rollcalls= roll_calls_clean.groupby('GeneralCourtNumber')['RollCallNumber'].apply(set).to_dict()

# Listing missing roll call numbers per general court
for i in max_rollcalls.items():
    court_no, max_rollcall= i[0], i[1]
    missing=[]
    for j in range(1,max_rollcall):
        if j not in court_rollcalls[court_no]:
            missing.append(j)
    if len(missing):
        print(f'General court: {court_no}, Missing roll calls: {missing}')



General court: 188, Missing roll calls: [1, 136, 147, 195, 385, 432, 467, 474]
General court: 189, Missing roll calls: [1, 95, 238, 299, 300, 301, 302, 303]
General court: 190, Missing roll calls: [1, 119, 314, 387, 452, 453, 454, 481]
General court: 191, Missing roll calls: [1, 93, 122, 321]
General court: 192, Missing roll calls: [1, 47]
General court: 193, Missing roll calls: [195]
General court: 194, Missing roll calls: [1]


## Bill-Rollcall mapping

In [26]:
bill_rollcalls.head()


,GeneralCourtNumber,BillNumber,RollCallNumber
0,186,H1696,293
1,186,H4156,348
2,186,H4316,295
3,186,H4368,291
4,186,H4470,190


In [27]:
bill_rollcalls.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2808 entries, 0 to 2807
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   GeneralCourtNumber  2808 non-null   int64 
 1   BillNumber          2808 non-null   object
 2   RollCallNumber      2808 non-null   int64 
dtypes: int64(2), object(1)
memory usage: 65.9+ KB


In [ ]:
bill_rollcalls_clean = bill_rollcalls.copy()

# Clean text fields in bill_rollcalls by converting to string, stripping whitespace, and replacing 'nan' with pd.NA
string_cols = ['BillNumber']
for col in string_cols:
    bill_rollcalls_clean[col] = (
        bill_rollcalls_clean[col]
        .astype(str)
        .str.strip()
        .replace({'nan': pd.NA, 'NaN': pd.NA})
    )

# Remove duplicate rows
bill_rollcalls_clean = bill_rollcalls_clean.drop_duplicates()
print(bill_rollcalls_clean.info())


<class 'pandas.core.frame.DataFrame'>
Index: 2804 entries, 0 to 2807
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   GeneralCourtNumber  2804 non-null   int64 
 1   BillNumber          2804 non-null   object
 2   RollCallNumber      2804 non-null   int64 
dtypes: int64(2), object(1)
memory usage: 87.6+ KB
None


In [29]:
bill_rollcalls_clean.to_csv('MA_bill_rollcalls.csv', index=False)

In [39]:
bill_rollcalls_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2804 entries, 0 to 2807
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   GeneralCourtNumber  2804 non-null   int64 
 1   BillNumber          2804 non-null   object
 2   RollCallNumber      2804 non-null   int64 
dtypes: int64(2), object(1)
memory usage: 87.6+ KB


# Final Denormalized dataset

In [27]:
bill_details_clean=pd.read_csv('../data/parsedcsv/MA_bill_details.csv')
bill_rollcalls_clean=pd.read_csv('../data/parsedcsv/MA_bill_rollcalls.csv')
legislature_members_clean= pd.read_csv('../data/parsedcsv/MA_legislature_members.csv')
roll_calls_clean=pd.read_csv('../data/parsedcsv/MA_senate_rollcalls.csv')

In [28]:
# Joining dataframes to create final rollcall dataframe

for df in [bill_details_clean, legislature_members_clean, roll_calls_clean]:
    df['GeneralCourtNumber'] = df['GeneralCourtNumber'].astype(int)

# Join rollcalls with members
rollcalls_members = roll_calls_clean.merge(
    legislature_members_clean,
    on=['GeneralCourtNumber', 'MemberCode'],
    how='left',
    validate='m:1'  
)

# Join rollcalls with bills to get BillId
rollcalls_bills = rollcalls_members.merge(
    bill_rollcalls_clean,
    on=['GeneralCourtNumber', 'RollCallNumber'],
    how='left',
    # validate='m:1'  
)

# Join with bill details
final_rollcall_df = rollcalls_bills.merge(
    bill_details_clean,
    on=['GeneralCourtNumber', 'BillNumber'],
    how='left',
    # validate='m:1'  
)

final_rollcall_df.drop(columns= ['Branch_y'], inplace=True)
final_rollcall_df.rename(columns={'Branch_x':'Branch'}, inplace=True)
final_rollcall_df.reset_index(drop=True, inplace=True)
final_rollcall_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98991 entries, 0 to 98990
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   GeneralCourtNumber  98991 non-null  int64 
 1   Branch              98991 non-null  object
 2   RollCallNumber      98991 non-null  int64 
 3   MemberCode          98811 non-null  object
 4   Vote                98991 non-null  object
 5   FullName            98811 non-null  object
 6   District            98811 non-null  object
 7   Party               98811 non-null  object
 8   LeadershipPosition  22198 non-null  object
 9   BillNumber          97905 non-null  object
 10  DocketNumber        11499 non-null  object
 11  PrimarySponsorId    64471 non-null  object
 12  PrimarySponsorName  95613 non-null  object
 13  Title               97905 non-null  object
dtypes: int64(2), object(12)
memory usage: 10.6+ MB


In [30]:
final_rollcall_df.head()

,GeneralCourtNumber,Branch,RollCallNumber,MemberCode,Vote,FullName,District,Party,LeadershipPosition,BillNumber,DocketNumber,PrimarySponsorId,PrimarySponsorName,Title
0,188,Senate,2,MJB0,Y,Michael J Barrett,Third Middlesex,Democrat,NaN,S7,NaN,S29,Senate Committee on Ethics and Rules,Senate Report recommending proposed Senate Rul...
1,188,Senate,2,SMB0,Y,Stephen M Brewer,Worcester Hampden Hampshire And Middlesex,Democrat,NaN,S7,NaN,S29,Senate Committee on Ethics and Rules,Senate Report recommending proposed Senate Rul...
2,188,Senate,2,WNB0,Y,William N Brownsberger,Second Suffolk And Middlesex,Democrat,NaN,S7,NaN,S29,Senate Committee on Ethics and Rules,Senate Report recommending proposed Senate Rul...
3,188,Senate,2,GDC0,Y,Gale D Candaras,First Hampden And Hampshire,Democrat,NaN,S7,NaN,S29,Senate Committee on Ethics and Rules,Senate Report recommending proposed Senate Rul...
4,188,Senate,2,HLC0,Y,Harriette L Chandler,First Worcester,Democrat,NaN,S7,NaN,S29,Senate Committee on Ethics and Rules,Senate Report recommending proposed Senate Rul...


In [31]:
# Missing rollcall and bill mapping
missing_bill = final_rollcall_df[final_rollcall_df['BillNumber'].isna()]
missing_summary = (
    missing_bill[['GeneralCourtNumber', 'RollCallNumber']]
    .drop_duplicates()
    .sort_values(['GeneralCourtNumber', 'RollCallNumber'])
)
len(missing_summary)

29

In [32]:
final_rollcall_df['BillNumber'].value_counts()

BillNumber
S3       7607
S4       7009
H3800    6725
H5164    5766
H3650    3684
         ... 
H4516      36
H3707      36
H4486      36
H140       35
H3564      34
Name: count, Length: 792, dtype: int64

In [33]:

# Group by GeneralCourtNumber and BillNumber, count rows
bill_counts = (
    bill_details_clean.groupby(['GeneralCourtNumber', 'BillNumber'])
      .size()
      .reset_index(name='RowCount')
)

print(bill_counts[bill_counts['RowCount']>1])


       GeneralCourtNumber BillNumber  RowCount
10449                 187      H3399         2
10672                 187        H36         2
13525                 187      S2071         2


In [34]:
final_rollcall_df.to_csv('MA_senate_rollcall_votes_denorm.csv', index=False)

## Committees

In [61]:

df = pd.read_csv('MA_committees.csv')

def filter_string_none(bills_list):
    if isinstance(bills_list, list):
        # Filter out elements that are exactly the string 'None'
        return [bill for bill in bills_list if bill != 'None']
    # Return non-list values as is (like the '0')
    return bills_list

# Apply the function to the 'Bills' column


df.rename(columns= {'CommitteName':'CommitteeName', 'CommitteShortName':'CommitteeShortName'},  inplace=True)
print(df.info())
#remove spaces in committee names 
df['CommitteeName']= df['CommitteeName'].str.replace(r'\s+', ' ', regex=True).str.strip()
df['CommitteeShortName']= df['CommitteeShortName'].str.replace(r'\s+', ' ', regex=True).str.strip()

df['Bills'] = df['Bills'].apply(filter_string_none)

df.drop_duplicates(subset=[c for c in df.columns if c not in ['Bills']], inplace=True)

print('Null values per column:')
print(df.isnull().sum())

print('Duplication in unique Id:', df[['CommitteeCode','GeneralCourtNumber']].duplicated().sum())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 636 entries, 0 to 635
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   CommitteeCode       636 non-null    object
 1   CommitteeName       636 non-null    object
 2   CommitteeShortName  636 non-null    object
 3   GeneralCourtNumber  636 non-null    int64 
 4   Branch              636 non-null    object
 5   SenateChairId       482 non-null    object
 6   HouseChairId        482 non-null    object
 7   Bills               636 non-null    object
dtypes: int64(1), object(7)
memory usage: 39.9+ KB
None
Null values per column:
CommitteeCode           0
CommitteeName           0
CommitteeShortName      0
GeneralCourtNumber      0
Branch                  0
SenateChairId         154
HouseChairId          154
Bills                   0
dtype: int64
Duplication in unique Id: 0


In [62]:
df

,CommitteeCode,CommitteeName,CommitteeShortName,GeneralCourtNumber,Branch,SenateChairId,HouseChairId,Bills
0,J27,Joint Committee on Transportation,Transportation,182,Joint,SAB0,JFW1,[]
1,J28,Joint Committee on Housing and Urban Development,Housing and Urban Development,182,Joint,HLC0,KGH1,[]
2,J37,Joint Committee on Energy,Energy,182,Joint,ROL0,JJB1,[]
3,J39,Joint committee on Ways and Means,Ways and Means (Joint),182,Joint,T M0,JHR1,[]
4,J40,Joint Rules,"Rules of the two branches, acting concurrently",182,Joint,SMB0,AMS1,[]
...,...,...,...,...,...,...,...,...
631,H34,House Committee on Ways and Means,Ways and Means (H),194,House of Representatives,NaN,AMM1,"['H51', 'H1', 'H55', 'H4003', 'H3431', 'H3346'..."
632,J50,Joint Committee on Cannabis Policy,Cannabis Policy (J),194,Joint,A_G0,DMD1,"['S75', 'S79', 'S82', 'S83', 'S85', 'S86', 'S8..."
633,TS10,Subcommittee on chapter 250 of the acts of 2024,Subcommittee on Rules,194,Senate,CFF0,NaN,"[None, None]"
634,J45,Joint Committee on Agriculture and Fisheries,Agriculture and Fisheries,194,Joint,DAF0,NMB1,"['S54', 'S55', 'S56', 'S58', 'S61', 'S62', 'S6..."


In [63]:
df.to_csv('MA_committees_cleaned.csv', index=False)

In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 632 entries, 0 to 631
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   CommitteeCode       632 non-null    object
 1   CommitteeName       632 non-null    object
 2   CommitteeShortName  632 non-null    object
 3   GeneralCourtNumber  632 non-null    int64 
 4   Branch              632 non-null    object
 5   SenateChairId       479 non-null    object
 6   HouseChairId        479 non-null    object
dtypes: int64(1), object(6)
memory usage: 34.7+ KB


In [23]:
reps = pd.read_csv('data/parsedcsv/MA_legislature_members.csv')
reps.head()

,GeneralCourtNumber,MemberCode,FullName,District,Branch,Party,LeadershipPosition
0,194,DAF0,Dylan A Fernandes,Plymouth And Barnstable,Senate,Democrat,NaN
1,194,KAD0,Kelly A Dooner,Third Bristol And Plymouth,Senate,Republican,Assistant Minority Leader
2,194,WJD0,William J Driscoll Jr,Norfolk Plymouth And Bristol,Senate,Democrat,NaN
3,194,MLB1,Michelle L Badger,1St Plymouth,House,Democrat,NaN
4,194,MSC1,Michael S Chaisson,1St Bristol,House,Republican,NaN


In [26]:
# Join committees with senate chairpersons
merged_df = pd.merge(
    df,
    reps[['GeneralCourtNumber', 'MemberCode', 'FullName']],
    left_on=['GeneralCourtNumber', 'SenateChairId'],
    right_on=['GeneralCourtNumber', 'MemberCode'],
    how='left',
    validate='m:1'  
)
merged_df.drop(columns=['MemberCode'], inplace=True)
merged_df.rename(columns={'FullName':'SenateChairPerson'}, inplace=True)
merged_df.info()

# Join committees with House chair persons
final_df = pd.merge(
    merged_df,
    reps[['GeneralCourtNumber', 'MemberCode', 'FullName']],
    left_on=['GeneralCourtNumber', 'HouseChairId'],
    right_on=['GeneralCourtNumber', 'MemberCode'],
    how='left',
    validate='m:1'  
)
final_df.drop(columns=['MemberCode'], inplace=True)
final_df.rename(columns={'FullName':'HouseChairPerson'}, inplace=True)
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 632 entries, 0 to 631
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   CommitteeCode       632 non-null    object
 1   CommitteeName       632 non-null    object
 2   CommitteeShortName  632 non-null    object
 3   GeneralCourtNumber  632 non-null    int64 
 4   Branch              632 non-null    object
 5   SenateChairId       479 non-null    object
 6   HouseChairId        479 non-null    object
 7   SenateChairPerson   479 non-null    object
dtypes: int64(1), object(7)
memory usage: 39.6+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 632 entries, 0 to 631
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   CommitteeCode       632 non-null    object
 1   CommitteeName       632 non-null    object
 2   CommitteeShortName  632 non-null    object
 3   GeneralCourtNumb

In [28]:
final_df.to_csv('MA_committees.csv', index=False)

In [36]:
final_rollcall_df.head()

,GeneralCourtNumber,Branch,RollCallNumber,MemberCode,Vote,FullName,District,Party,LeadershipPosition,BillNumber,DocketNumber,PrimarySponsorId,PrimarySponsorName,Title
0,188,Senate,2,MJB0,Y,Michael J Barrett,Third Middlesex,Democrat,NaN,S7,NaN,S29,Senate Committee on Ethics and Rules,Senate Report recommending proposed Senate Rul...
1,188,Senate,2,SMB0,Y,Stephen M Brewer,Worcester Hampden Hampshire And Middlesex,Democrat,NaN,S7,NaN,S29,Senate Committee on Ethics and Rules,Senate Report recommending proposed Senate Rul...
2,188,Senate,2,WNB0,Y,William N Brownsberger,Second Suffolk And Middlesex,Democrat,NaN,S7,NaN,S29,Senate Committee on Ethics and Rules,Senate Report recommending proposed Senate Rul...
3,188,Senate,2,GDC0,Y,Gale D Candaras,First Hampden And Hampshire,Democrat,NaN,S7,NaN,S29,Senate Committee on Ethics and Rules,Senate Report recommending proposed Senate Rul...
4,188,Senate,2,HLC0,Y,Harriette L Chandler,First Worcester,Democrat,NaN,S7,NaN,S29,Senate Committee on Ethics and Rules,Senate Report recommending proposed Senate Rul...


In [ ]:
# Checking for duplicate member votes in the final rollcall dataframe
dupe_member_votes = (
    final_rollcall_df
    .groupby(['GeneralCourtNumber', 'RollCallNumber', 'MemberCode'])
    .size()
    .reset_index(name='count')
)

# Filter to show only duplicates
dupe_member_votes = dupe_member_votes[dupe_member_votes['count'] > 1]
print("Duplicate member votes:")
print(dupe_member_votes)

Duplicate member votes:
Empty DataFrame
Columns: [GeneralCourtNumber, RollCallNumber, MemberCode, count]
Index: []


In [37]:
leadership_to_check = [
    'Speaker of the House',
    'President Pro Tempore'
]

leadership_filtered = final_rollcall_df[final_rollcall_df['LeadershipPosition'].isin(leadership_to_check)]

# Count how many members are assigned each leadership position in each roll call
leadership_conflicts = (
    leadership_filtered
    .groupby(['GeneralCourtNumber', 'RollCallNumber', 'LeadershipPosition'])
    .MemberCode
    .nunique()
    .reset_index(name='unique_members')
)

# Filter to keep only problematic ones
leadership_conflicts = leadership_conflicts[leadership_conflicts['unique_members'] > 1]

print("Conflicting leadership positions (should be unique):")
print(leadership_conflicts)


Conflicting leadership positions (should be unique):
Empty DataFrame
Columns: [GeneralCourtNumber, RollCallNumber, LeadershipPosition, unique_members]
Index: []
